In [1]:
# Cell 1
import sys
sys.path.append('../')
from dotenv import load_dotenv
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)
embedder = SentenceTransformer(
    'sentence-transformers/'
    'paraphrase-multilingual-MiniLM-L12-v2'
)
print("Model loaded")

e:\graph-rag-compliance\venv\Lib\site-packages\sentence_transformers\cross_encoder\CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Model loaded


In [2]:
# Cell 2 — Embed tất cả Activity nodes
with driver.session() as session:
    result = session.run("""
        MATCH (a:Activity)
        RETURN a.id AS id, a.name AS name,
               a.description AS description,
               a.article_ref AS article_ref
    """)
    activities = [dict(r) for r in result]

print(f"Cần embed {len(activities)} Activity nodes")

for act in activities:
    text = (
        f"{act['name']}. "
        f"{act['description']}. "
        f"{act['article_ref']}"
    )
    embedding = embedder.encode(text).tolist()

    with driver.session() as session:
        session.run("""
            MATCH (a:Activity {id: $id})
            SET a.embedding = $embedding
        """, id=act['id'], embedding=embedding)

    print(f"  ✓ {act['name']}")

print("\nXong — tất cả Activity đã có embedding")

Cần embed 13 Activity nodes
  ✓ A_Create Application
  ✓ A_Submitted
  ✓ A_Concept
  ✓ A_Accepted
  ✓ O_Create Offer
  ✓ O_Created
  ✓ O_Sent (mail and online)
  ✓ A_Complete
  ✓ O_Accepted
  ✓ O_Returned
  ✓ A_Cancelled
  ✓ A_Incomplete
  ✓ W_Call incomplete files

Xong — tất cả Activity đã có embedding


In [3]:
# Cell 3 — Tạo vector index trong Neo4j
load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

with driver.session() as session:
    session.run("""
        CREATE VECTOR INDEX activity_embedding
        IF NOT EXISTS
        FOR (a:Activity) ON (a.embedding)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 384,
            `vector.similarity_function`: 'cosine'
        }}
    """)
    print("Vector index created")

driver.close()

Vector index created


In [4]:
# Cell debug — kiểm tra embedding
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

with driver.session() as session:
    # Kiểm tra có embedding chưa
    r1 = session.run("""
        MATCH (a:Activity)
        WHERE a.embedding IS NOT NULL
        RETURN count(a) AS has_embedding
    """).single()

    r2 = session.run("""
        MATCH (a:Activity)
        WHERE a.embedding IS NULL
        RETURN count(a) AS no_embedding
    """).single()

    # Kiểm tra vector index
    r3 = session.run("""
        SHOW INDEXES
        WHERE type = 'VECTOR'
    """)
    indexes = [dict(r) for r in r3]

print(f"Có embedding : {r1['has_embedding']}")
print(f"Không embedding: {r2['no_embedding']}")
print(f"Vector indexes: {len(indexes)}")
for idx in indexes:
    print(f"  {idx.get('name')} — state: {idx.get('state')}")

driver.close()

Có embedding : 13
Không embedding: 0
Vector indexes: 1
  activity_embedding — state: ONLINE


In [9]:
# Cell debug 2 — test vector search trực tiếp
from sentence_transformers import SentenceTransformer
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

embedder = SentenceTransformer(
    'sentence-transformers/'
    'paraphrase-multilingual-MiniLM-L12-v2'
)

query     = "mandatory activity A_Complete required Article 10(1)"
query_vec = embedder.encode(query).tolist()
print(f"Vector dimensions: {len(query_vec)}")

with driver.session() as session:
    # Thử vector search
    try:
        result = session.run("""
            CALL db.index.vector.queryNodes(
                'activity_embedding',
                3,
                $query_vec
            ) YIELD node, score
            RETURN node.name AS name, score
            ORDER BY score DESC
        """, query_vec=query_vec)
        nodes = [dict(r) for r in result]
        print(f"\nVector search kết quả:")
        for n in nodes:
            print(f"  {n['name']:<40} score={n['score']:.4f}")
    except Exception as e:
        print(f"Vector search lỗi: {e}")

driver.close()

Vector dimensions: 384

Vector search kết quả:
  A_Incomplete                             score=0.7652
  A_Complete                               score=0.7550
  A_Submitted                              score=0.7442


In [8]:
# Cell chờ index ONLINE
import time
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

print("Đang chờ index ONLINE...")
while True:
    with driver.session() as session:
        result = session.run("""
            SHOW INDEXES
            WHERE name = 'activity_embedding'
        """).single()
        state = result['state']
        print(f"  State: {state}")
        if state == 'ONLINE':
            print("Index ONLINE — sẵn sàng!")
            break
    time.sleep(3)

driver.close()

Đang chờ index ONLINE...
  State: ONLINE
Index ONLINE — sẵn sàng!


In [6]:
# Cell fix index
from dotenv import load_dotenv
from neo4j import GraphDatabase
import os

load_dotenv('../.env')
driver = GraphDatabase.driver(
    os.getenv("NEO4J_URI"),
    auth=(os.getenv("NEO4J_USERNAME"),
          os.getenv("NEO4J_PASSWORD"))
)

with driver.session() as session:
    session.run("""
        CREATE VECTOR INDEX activity_embedding
        IF NOT EXISTS
        FOR (a:Activity) ON (a.embedding)
        OPTIONS {indexConfig: {
            `vector.dimensions`: 384,
            `vector.similarity_function`: 'cosine'
        }}
    """)
    print("Vector index 384 chiều tạo xong")

    # Kiểm tra
    result = session.run("""
        SHOW INDEXES
        WHERE type = 'VECTOR'
    """)
    for r in result:
        print(f"  {r['name']} — state: {r['state']}")

driver.close()

Vector index 384 chiều tạo xong
  activity_embedding — state: POPULATING
